In [2]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Membuat SparkSession
spark = SparkSession.builder \
    .appName("Tugas11-MLlibRegresi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

# Membaca dataset tugas11_penjualan.csv yang telah dibuat
df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)

print(f"Dataset berhasil dimuat: {df.count()} baris")
df.show(5)

SparkSession siap. Versi Spark: 4.2.0
Dataset berhasil dimuat: 2500 baris
+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows


In [6]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

# StringIndexer untuk kolom kategorikal
indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

# Terapkan StringIndexer secara berurutan
df_indexed = indexer_menu.fit(df).transform(df)
df_indexed = indexer_cabang.fit(df_indexed).transform(df_indexed)
df_indexed = indexer_waktu.fit(df_indexed).transform(df_indexed)

# VectorAssembler untuk menggabungkan seluruh fitur menjadi satu kolom vector
assembler_utama = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)

df_final = assembler_utama.transform(df_indexed)
df_final.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "features", "total_penjualan").show(5, truncate=False)

+--------------+----------+-----------+---+------------+-------------------------+------------------+
|nama_menu     |cabang    |waktu_pesan|qty|harga_satuan|features                 |total_penjualan   |
+--------------+----------+-----------+---+------------+-------------------------+------------------+
|Kentang Goreng|Semarang  |Malam      |1  |30000       |[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|Kentang Goreng|Yogyakarta|Sore       |3  |20000       |[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|Kentang Goreng|Semarang  |Siang      |5  |22000       |[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|Kentang Goreng|Yogyakarta|Pagi       |2  |20000       |[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|Roti Bakar    |Magelang  |Malam      |2  |30000       |[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
+--------------+----------+-----------+---+------------+-------------------------+------------------+
only showing top 5 rows


In [11]:
# bagi data jadi 75% training 25% testing dengan seed=101
train, test = df_features.randomSplit([0.75, 0.25], seed=101)

print(f"Jumlah data training : {train.count()}")
print(f"Jumlah data testing   : {test.count()}")

from pyspark.ml.regression import LinearRegression

# latih model Linear Regression dengan labelCol="total_penjualan".
lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model = lr.fit(train)

print("Model berhasil dilatih!")
print("Koefisien tiap fitur:", model.coefficients)
print("Intercept:", model.intercept)

Jumlah data training : 1839
Jumlah data testing   : 661
Model berhasil dilatih!
Koefisien tiap fitur: [22046.663271025755,3.0229208681606696,-123.21425598657828,279.2845246167609,339.63879532317213]
Intercept: -66903.1776648588


In [12]:
from pyspark.ml.evaluation import RegressionEvaluator

# prediksi pada data testing
predictions = model.transform(test)

# menghitung RMSE (Root Mean Squared Error)
evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(predictions)

# menghitung R squared (R²)
evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f}  ({r2*100:.1f}% variasi data berhasil dijelaskan model)")

RMSE : Rp 10,901
R²   : 0.9107  (91.1% variasi data berhasil dijelaskan model)


In [13]:
# Model Kedua(hanya menggunakan fitur qty dan harga_satuan)
assembler_kedua = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_kedua"
)

df_kedua = assembler_kedua.transform(df)
train2, test2 = df_kedua.randomSplit([0.75, 0.25], seed=101)

lr2 = LinearRegression(featuresCol="features_kedua", labelCol="total_penjualan")
model_kedua = lr2.fit(train2)

predictions2 = model_kedua.transform(test2)
r2_kedua = evaluator_r2.evaluate(predictions2)

print(f"R² Model Utama (5 Fitur)   : {r2:.4f}")
print(f"R² Model Kedua (2 Fitur)   : {r2_kedua:.4f}")

R² Model Utama (5 Fitur)   : 0.9107
R² Model Kedua (2 Fitur)   : 0.9108


In [17]:
import pandas as pd

# membuat DataFrame pandas berisi 3 skenario
data_baru_pd = pd.DataFrame({
    "nama_menu": ["Kopi Susu", "Americano", "Roti Bakar"],
    "cabang": ["Magelang", "Yogyakarta", "Semarang"],
    "waktu_pesan": ["Pagi", "Siang", "Malam"],
    "qty": [3, 1, 2],
    "harga_satuan": [15000, 20000, 25000]
})

df_baru = spark.createDataFrame(data_baru_pd)

# menggunakan model StringIndexer yang udah di fit sebelumnya
fitted_indexer_menu = indexer_menu.fit(df)
fitted_indexer_cabang = indexer_cabang.fit(df)
fitted_indexer_waktu = indexer_waktu.fit(df)

df_baru_indexed = fitted_indexer_menu.transform(df_baru)
df_baru_indexed = fitted_indexer_cabang.transform(df_baru_indexed)
df_baru_indexed = fitted_indexer_waktu.transform(df_baru_indexed)

# transformasi menggunakan VectorAssembler utama
df_baru_features = assembler_utama.transform(df_baru_indexed)

# mlakukan prediksi total_penjualan
pred_baru = model_utama.transform(df_baru_features)

# menampilkan hasil prediksi untuk skenario baru
print("Hasil Prediksi Omzet untuk 3 Skenario Baru:")
pred_baru.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction").show(truncate=False)

Hasil Prediksi Omzet untuk 3 Skenario Baru:
+----------+----------+-----------+---+------------+------------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction        |
+----------+----------+-----------+---+------------+------------------+
|Kopi Susu |Magelang  |Pagi       |3  |15000       |44457.410914641936|
|Americano |Yogyakarta|Siang      |1  |20000       |16560.465084643452|
|Roti Bakar|Semarang  |Malam      |2  |25000       |53168.52140181983 |
+----------+----------+-----------+---+------------+------------------+



In [ ]:
spark.stop()
print("SparkSession ditutup.")